# Notebook 6 — Envío de predicciones a Beckhoff (ADS)
## Del clasificador al brazo robótico

Último paso del lab de **Visión**. Toma el modelo de *Transfer Learning* entrenado en el
**NB4**, clasifica la cámara en vivo y **envía el resultado al IPC Beckhoff por ADS**.

**Contrato de comunicación (3 BOOLs):**

| Color detectado | Variable PLC | Acción del brazo |
|---|---|---|
| rojo | `VARIABLES.bRojo` | deposita en zona roja |
| azul | `VARIABLES.bAzul` | deposita en zona azul |
| amarillo | `VARIABLES.bAmarillo` | deposita en zona amarilla |
| fondo / baja confianza | *(ninguna)* | el brazo no actúa |

Si la confianza supera el umbral, se escribe `True` en el BOOL del color y `False` en los
otros dos. El PLC detecta el flanco y dispara la salida digital del brazo.

| Paso | Descripción |
|------|-------------|
| 1 | Setup |
| 2 | Cargar el modelo entrenado (NB4) |
| 3 | Configuración ADS + mapeo de clases a BOOLs |
| 4 | Conexión con el PLC *(completar)* |
| 5 | Escritura de los BOOLs *(completar)* |
| 6 | Bucle principal: cámara → predicción → envío |

> **Requisitos:** un modelo en `modelos/` (entrénelo en el NB4), `pyads` instalado
> (`pip install pyads`) y la GVL `VARIABLES` declarada en TwinCAT con `bRojo`, `bAzul`,
> `bAmarillo`. Sin PLC, el bucle corre en **modo simulación** (imprime lo que escribiría).

---
## 1. Setup

In [ ]:
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')   # ocultar logs de TensorFlow
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '-1')  # usar CPU
import os, glob, time
import json as _json
from collections import deque
import numpy as np
import cv2
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
from tensorflow.keras.models import load_model
import ipywidgets as widgets
from IPython.display import display, clear_output

# pyads es opcional: si no está, el notebook corre en modo simulación
try:
    import pyads
    PYADS_OK = True
except ImportError:
    PYADS_OK = False

cfg = {'model': None, 'class_names': [], 'img_size': 128, 'preprocessing': 'rescale', 'model_path': ''}

def _detectar_preprocesamiento(model):
    """Detecta si el modelo usa MobileNetV2."""
    for layer in model.layers:
        if 'mobilenet' in layer.name.lower():
            return 'mobilenet'
    return 'rescale'

def _preprocess(img_array, preprocessing):
    """Preprocesa según el tipo de modelo (igual que en el NB5)."""
    if preprocessing == 'mobilenet':
        from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
        return preprocess_input(img_array.astype('float32'))
    return img_array.astype('float32') / 255.0

print(f'TensorFlow {tf.__version__} — Listo.  pyads: ' + ('disponible' if PYADS_OK else 'NO instalado (modo simulación)'))

---
## 2. Cargar el modelo entrenado

Seleccione el modelo de *Transfer Learning* generado en el **NB4** (busca en `modelos/`).
Las clases y el preprocesamiento se leen automáticamente del `.json`.

In [ ]:
# --- Descubrir modelos disponibles ---
_modelos = {}
for d in ['modelos', '../modelos', '.']:
    if not os.path.isdir(d):
        continue
    for h5 in glob.glob(os.path.join(d, '*.h5')) + glob.glob(os.path.join(d, '*.keras')):
        jp = h5.rsplit('.', 1)[0] + '.json'
        meta = _json.load(open(jp)) if os.path.exists(jp) else {}
        va = meta.get('val_accuracy')
        label = os.path.basename(h5) + (f'  (val {va:.0%})' if va is not None else '')
        _modelos[label] = {'path': h5, 'meta': meta}

def _cargar_modelo(model_path):
    model = load_model(model_path, compile=False)
    meta = {}
    jp = model_path.rsplit('.', 1)[0] + '.json'
    if os.path.exists(jp):
        meta = _json.load(open(jp))
    cfg['model'] = model
    cfg['img_size'] = meta.get('img_size', model.input_shape[1])
    cfg['preprocessing'] = meta.get('preprocessing', _detectar_preprocesamiento(model))
    cfg['class_names'] = meta.get('class_names', [f'Clase {i}' for i in range(model.output_shape[-1])])
    cfg['model_path'] = model_path
    print(f'✔ Modelo: {os.path.basename(model_path)}')
    print(f'  Clases: {cfg["class_names"]}')
    print(f'  Input: {cfg["img_size"]}px | Preprocesamiento: {cfg["preprocessing"]}')

if _modelos:
    _dd = widgets.Dropdown(options=list(_modelos.keys()), description='Modelo:',
                           layout=widgets.Layout(width='100%'),
                           style={'description_width': 'initial'})
    _btn = widgets.Button(description=' Cargar', button_style='success', icon='check')
    _out = widgets.Output()
    def _on(_):
        with _out:
            clear_output(wait=True)
            _cargar_modelo(_modelos[_dd.value]['path'])
    _btn.on_click(_on)
    display(widgets.VBox([_dd, _btn, _out]))
    print('Seleccione un modelo y presione Cargar.')
else:
    print('No se encontraron modelos. Entrene uno primero en notebook4_transfer_learning.ipynb')

---
## 3. Configuración ADS y mapeo de clases

Ajuste el **AMS Net ID** de su IPC y el **umbral de confianza**. El mapeo conecta cada
clase del modelo con su variable `BOOL` en la GVL del PLC. Las clases sin entrada en el
mapeo (p.ej. `fondo`) no disparan ninguna acción.

In [ ]:
# ── Configuración ADS (Beckhoff / TwinCAT 3) ──
AMS_NET_ID = '5.80.201.232.1.1'   # <-- AMS Net ID de SU IPC Beckhoff
ADS_PORT   = 851                  # 851 = TwinCAT 3 Runtime 1
UMBRAL_CONFIANZA = 0.85           # confianza mínima para activar un BOOL

# Mapeo clase del modelo -> variable BOOL en la GVL 'VARIABLES'.
# Las clases que NO aparezcan aquí (p.ej. 'fondo') no activan ninguna salida.
VAR_POR_CLASE = {
    'rojo':     'VARIABLES.bRojo',
    'azul':     'VARIABLES.bAzul',
    'amarillo': 'VARIABLES.bAmarillo',
}

# Avisar si alguna clase del modelo no está mapeada (informativo)
if cfg['class_names']:
    sin_mapeo = [c for c in cfg['class_names'] if c not in VAR_POR_CLASE]
    print('Clases del modelo :', cfg['class_names'])
    print('Clases con BOOL   :', list(VAR_POR_CLASE.keys()))
    print('Sin acción (fondo):', sin_mapeo)
else:
    print('⚠ Cargue un modelo en el paso 2 antes de continuar.')

---
## 4. Conexión con el PLC  *(completar)*

Complete `conectar_plc()` con `pyads`. Mientras no lo complete, el notebook funciona en
**modo simulación**: el bucle del paso 6 corre e imprime lo que escribiría, sin tocar el PLC.

In [ ]:
plc = {'conn': None, 'connected': False}

def conectar_plc(ams_net_id=AMS_NET_ID, port=ADS_PORT):
    """Abre la conexión ADS con el IPC Beckhoff."""
    if not PYADS_OK:
        print('⚠ pyads no instalado. Ejecute: pip install pyads  (se usará modo simulación)')
        return False
    try:
        # ───── TODO ALUMNO: descomentar para conectar de verdad ─────
        # conn = pyads.Connection(ams_net_id, port)
        # conn.open()
        # plc['conn'] = conn
        # plc['connected'] = True
        # print(f'✔ Conectado a {ams_net_id}:{port}')
        # return True
        # ─────────────────────────────────────────────────────────────

        # Placeholder mientras no se completa (modo simulación):
        plc['connected'] = False
        print('⚠ TODO: complete conectar_plc(). El bucle correrá en MODO SIMULACIÓN.')
        return False
    except Exception as e:
        plc['connected'] = False
        print(f'✘ Error de conexión ADS: {e}')
        return False

def desconectar_plc():
    if plc['conn'] is not None and plc['connected']:
        try:
            # TODO ALUMNO: plc['conn'].close()
            pass
        except Exception:
            pass
    plc['connected'] = False
    plc['conn'] = None

conectar_plc()

---
## 5. Escritura de los BOOLs  *(completar)*

`escribir_bools()` activa el BOOL del color detectado (si supera el umbral) y apaga los
otros dos. Complete la llamada `write_by_name`. Devuelve un `dict {variable: valor}` para
poder mostrar el estado en pantalla aunque no haya PLC.

In [ ]:
def escribir_bools(clase, confianza):
    """Activa el BOOL de la clase detectada y apaga los demás. Devuelve {var: valor}."""
    activa = clase if (confianza >= UMBRAL_CONFIANZA and clase in VAR_POR_CLASE) else None
    escrito = {}
    for cls, var in VAR_POR_CLASE.items():
        valor = (cls == activa)
        escrito[var] = valor
        if plc['connected']:
            # ───── TODO ALUMNO: descomentar para escribir al PLC ─────
            # plc['conn'].write_by_name(var, valor, pyads.PLCTYPE_BOOL)
            # ─────────────────────────────────────────────────────────
            pass
    return escrito

def apagar_todo():
    """Pone los tres BOOLs en False (al salir del bucle)."""
    for var in VAR_POR_CLASE.values():
        if plc['connected']:
            # TODO ALUMNO: plc['conn'].write_by_name(var, False, pyads.PLCTYPE_BOOL)
            pass

# Prueba rápida (sin cámara): qué se escribiría para una detección 'rojo' al 0.92
print('Ejemplo escribir_bools("rojo", 0.92):')
for k, v in escribir_bools('rojo', 0.92).items():
    print(f'  {k} = {v}')

---
## 6. Bucle principal: cámara → predicción → envío

Abre la cámara, clasifica cada frame (con suavizado temporal anti-parpadeo) y envía el
resultado al PLC. En pantalla se ve la clase, la confianza y el estado de los tres BOOLs.

**Controles:** `Q` salir. Al salir, los tres BOOLs se ponen en `False` y se cierra la conexión.

> En **WSL2** no hay acceso a la cámara USB. Ejecute este notebook desde **Windows**
> (PowerShell + VS Code), tal como lo hará en el laboratorio.

In [ ]:
CAMERA_INDEX = 0          # cambiar a 1 si no detecta la cámara
SMOOTHING_WINDOW = 5      # promedio de las últimas N predicciones (anti-parpadeo)

model = cfg['model']
names = cfg['class_names']
sz = cfg['img_size']
preprocessing = cfg['preprocessing']

COLOR_BOOL = {'VARIABLES.bRojo': (60, 60, 230),
              'VARIABLES.bAzul': (230, 150, 40),
              'VARIABLES.bAmarillo': (40, 210, 230)}

def draw_overlay(frame, clase, conf, escrito, fps):
    h, w = frame.shape[:2]
    ov = frame.copy()
    cv2.rectangle(ov, (0, 0), (w, 56), (30, 30, 30), -1)
    cv2.addWeighted(ov, 0.85, frame, 0.15, 0, frame)
    activo = conf >= UMBRAL_CONFIANZA and clase in VAR_POR_CLASE
    label = f'{clase}  {conf*100:.0f}%' if activo else f'{clase}?  {conf*100:.0f}%'
    col = (0, 230, 0) if activo else (150, 150, 150)
    cv2.putText(frame, label, (14, 38), cv2.FONT_HERSHEY_SIMPLEX, 1.0, col, 2, cv2.LINE_AA)
    modo = 'PLC' if plc['connected'] else 'SIM'
    cv2.putText(frame, f'{modo}  {fps:.0f}fps', (w - 150, 36),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (180, 180, 180), 1, cv2.LINE_AA)
    # estado de los 3 BOOLs
    y = 86
    for var, val in escrito.items():
        c = COLOR_BOOL.get(var, (200, 200, 200)) if val else (90, 90, 90)
        cv2.circle(frame, (24, y - 5), 9, c, -1)
        cv2.putText(frame, f'{var} = {val}', (44, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, c, 2 if val else 1, cv2.LINE_AA)
        y += 32
    cv2.putText(frame, 'Q = salir', (10, h - 12),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (140, 140, 140), 1, cv2.LINE_AA)
    return frame

def _es_wsl():
    try:
        return 'microsoft' in open('/proc/version').read().lower()
    except Exception:
        return False

if model is None:
    print('✘ Cargue un modelo primero (paso 2).')
elif _es_wsl():
    print('⚠ WSL2 no accede a la cámara USB. Ejecute este notebook desde Windows.')
else:
    cap = cv2.VideoCapture(CAMERA_INDEX)
    if not cap.isOpened():
        print(f'✘ No se pudo abrir la cámara (índice {CAMERA_INDEX}).')
    else:
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
        buf = deque(maxlen=SMOOTHING_WINDOW)
        prev = 0
        print(f'Cámara abierta. Modo: ' + ('PLC' if plc['connected'] else 'SIMULACIÓN') + ". 'Q' para salir.")
        try:
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                now = time.time()
                fps = 1 / (now - prev) if prev else 0
                prev = now
                img = cv2.cvtColor(cv2.resize(frame, (sz, sz)), cv2.COLOR_BGR2RGB)
                pred = model.predict(np.expand_dims(_preprocess(img, preprocessing), 0), verbose=0)[0]
                buf.append(pred)
                p = np.mean(buf, axis=0)
                idx = int(np.argmax(p))
                clase, conf = names[idx], float(p[idx])
                escrito = escribir_bools(clase, conf)
                frame = draw_overlay(frame, clase, conf, escrito, fps)
                cv2.imshow('PUCP - Inferencia + Beckhoff (ADS)', frame)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
        except KeyboardInterrupt:
            print('Interrumpido.')
        finally:
            apagar_todo()
            desconectar_plc()
            cap.release()
            cv2.destroyAllWindows()
            print('Cámara cerrada y BOOLs en False.')